# Module 14: Async Human Approval - 01: A queue, a suspended graph, a separate reviewer

> **MLCourse - Agentic AI - Agent Patterns**

[`02_langgraph/04_human_in_the_loop`](../../02_langgraph/04_human_in_the_loop/README.md)
covered `interrupt()`: the graph pauses, a human answers *right there in the
same notebook cell*, and the graph resumes. That works when a person is at
the keyboard, watching. It does not model the far more common real case: an
approval request goes out, and the human looks at it **whenever they get to
it** - an hour later, the next morning, after a weekend.

This module is that version. The agent writes a request to a durable **queue**
(a JSON file, standing in for a real message queue or database table), then
suspends via the LangGraph checkpointing you already know from
[`02_langgraph/03_persistence_checkpointing`](../../02_langgraph/03_persistence_checkpointing/README.md).
A *separate* notebook cell - standing in for a reviewer's dashboard running
in an entirely different process, possibly on a different machine, possibly
days later - reads the queue, decides, and writes the answer back. Only then
does the original graph resume.

### What you will learn in this module

1. Why "the human is at the keyboard" is the wrong default for approvals that
   matter.
2. The queue-write / suspend / separate-reviewer-cell / resume pattern.
3. Timeout and escalation policy (notebook 02).
4. Multi-reviewer sign-off - N-of-M approval (notebook 03).

### Scope, stated plainly

**No web server anywhere in this module.** A production system would put a
real HTTP endpoint or a Slack app in front of the reviewer side; that is
deployment, and it is out of scope by design (this course covers it
separately, in end-to-end projects). Everything here is a JSON file on disk
and ordinary Python function calls, which is enough to demonstrate the
*pattern* faithfully: durable request, real suspension, decoupled reviewer.

No LLM calls, no API key - this module is about orchestration and state, not
generation.

### Setup


In [ ]:
import json
import os
import sqlite3
import time
import uuid
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Optional, TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

QUEUE_PATH = Path("approval_queue.json")
DB_PATH = "async_approval.db"

if QUEUE_PATH.exists():
    QUEUE_PATH.unlink()
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

checkpointer = SqliteSaver(sqlite3.connect(DB_PATH, check_same_thread=False))

print("Module 14: Async Human Approval")
print("No API key required -- this module is pure orchestration.")


### 1. Why "the human is watching" is the wrong default

`04_human_in_the_loop`'s pattern is: call `.invoke(...)`, the call blocks,
`interrupt()` fires, you (in the very next cell) provide the answer, and the
same Python call that started the run receives it. That is exactly right for
debugging, and for interactive tools where a person genuinely is sitting
there.

It breaks down the moment the approval represents **real-world latency**:

- A purchase over a threshold needs a manager's sign-off, and the manager is
  in a meeting.
- A legal document needs a lawyer's review, and lawyers do not sit refreshing
  a terminal.
- A code change needs a second engineer's approval, and it is 2am where they
  are.

In every one of these, the process that *asked* the question cannot stay
alive waiting for the answer - it might be a serverless function that timed
out minutes ago, or a notebook kernel someone restarted. **The suspended
state has to survive the asking process's death**, and the answer has to be
able to arrive from a completely different process, arbitrarily later.

That is precisely what a checkpointer gives you (you built this muscle in
[`02_langgraph/03`](../../02_langgraph/03_persistence_checkpointing/README.md)
and again in
[`02_langgraph/10_state_migration_and_versioning`](../../02_langgraph/10_state_migration_and_versioning/README.md),
which is about exactly this kind of long-lived suspended thread meeting a
schema change). What this module adds is the **notification** half: how does
the reviewer even find out there is something to approve, if they are not
watching a terminal?

### 2. The pattern

```
AGENT PROCESS                    DURABLE STORE               REVIEWER PROCESS
--------------                   ------------                ----------------
1. build request        ---->    approval_queue.json
2. graph.invoke()
   -> interrupt() fires
   -> checkpoint written,
      graph SUSPENDED,
      this process can
      now exit or crash;
      nothing is lost

                                                          (arbitrarily later --
                                                           minutes, hours, days)

                                  approval_queue.json <----  3. reviewer reads
                                                              the pending request
                                  answer written        <----  4. reviewer decides

5. a DIFFERENT invocation
   of Command(resume=...)
   against the SAME
   thread_id picks the
   checkpoint back up
   and completes
```

The essential property: **steps 1-2 and steps 3-5 do not need to be the same
process, the same machine, or close together in time.** All that connects
them is the `thread_id` and the durable store. Let's build it.

### 3. The queue

A minimal durable queue: a JSON file holding pending requests, keyed by an id.
In production this is a database table, an SQS queue, or a Slack message with
buttons - the interface (`enqueue` / `read pending` / `record answer`) is
what matters, not the storage.

### A minimal durable approval queue


In [ ]:
@dataclass
class ApprovalRequest:
    request_id: str
    thread_id: str
    summary: str
    payload: dict
    created_at: float
    status: str = "pending"          # pending | approved | rejected
    decided_by: Optional[str] = None
    decided_at: Optional[float] = None
    reason: Optional[str] = None


class ApprovalQueue:
    """A JSON-file-backed queue. Stands in for a real message queue or
    database table -- the interface is what this module is teaching, not
    the storage engine."""

    def __init__(self, path: Path):
        self.path = path
        if not self.path.exists():
            self._write({})

    def _read(self) -> dict:
        return json.loads(self.path.read_text()) if self.path.exists() else {}

    def _write(self, data: dict) -> None:
        self.path.write_text(json.dumps(data, indent=2))

    def enqueue(self, thread_id: str, summary: str, payload: dict) -> str:
        req = ApprovalRequest(request_id=str(uuid.uuid4())[:8], thread_id=thread_id,
                              summary=summary, payload=payload, created_at=time.time())
        data = self._read()
        data[req.request_id] = asdict(req)
        self._write(data)
        return req.request_id

    def pending(self) -> list:
        return [r for r in self._read().values() if r["status"] == "pending"]

    def decide(self, request_id: str, approved: bool, reviewer: str, reason: str = "") -> None:
        data = self._read()
        data[request_id]["status"] = "approved" if approved else "rejected"
        data[request_id]["decided_by"] = reviewer
        data[request_id]["decided_at"] = time.time()
        data[request_id]["reason"] = reason
        self._write(data)

    def get(self, request_id: str) -> dict:
        return self._read()[request_id]


queue = ApprovalQueue(QUEUE_PATH)
print(f"queue file: {QUEUE_PATH.resolve()}")


### 4. The agent side: write to the queue, then suspend

The node that needs approval does two things, in this order: it **enqueues**
a durable request describing what needs a decision, and only then calls
`interrupt()`. The order matters - if you called `interrupt()` first, a crash
between the interrupt and the enqueue would leave the graph suspended with no
request the reviewer could ever find, a wedged thread with no way out.

### The graph: a purchase that needs approval over a threshold


In [ ]:
class PurchaseState(TypedDict):
    item: str
    amount: float
    request_id: str
    verdict: str


def prepare(state: PurchaseState) -> dict:
    item, amount = state["item"], state["amount"]
    print(f"  [prepare] validating purchase of {item} (${amount:.2f})")
    return {}


def request_approval(state: PurchaseState) -> dict:
    """Enqueue FIRST, interrupt SECOND. If this process dies between the two
    lines, the worst case is a request nobody answers -- not a suspended
    thread nobody can find."""
    thread_id = "purchase-in-progress"     # in a real node, read this from config
    req_id = queue.enqueue(
        thread_id=thread_id,
        summary=f"Approve purchase: {state['item']} for ${state['amount']:.2f}",
        payload={"item": state["item"], "amount": state["amount"]},
    )
    print(f"  [request_approval] enqueued request {req_id}")

    # interrupt() suspends here. Its payload is what a reviewer UI would show;
    # we ALSO durably stored the same information in the queue a moment ago,
    # which is what lets the reviewer find this without ever touching the
    # graph process directly.
    verdict = interrupt({"request_id": req_id, "summary": queue.get(req_id)["summary"]})
    return {"request_id": req_id, "verdict": verdict}


def finalize(state: PurchaseState) -> dict:
    outcome = "APPROVED" if state["verdict"] == "approved" else "REJECTED"
    print(f"  [finalize] purchase {outcome}: {state['item']}")
    return {"verdict": outcome}


builder = StateGraph(PurchaseState)
builder.add_node("prepare", prepare)
builder.add_node("request_approval", request_approval)
builder.add_node("finalize", finalize)
builder.add_edge(START, "prepare")
builder.add_edge("prepare", "request_approval")
builder.add_edge("request_approval", "finalize")
builder.add_edge("finalize", END)

purchase_agent = builder.compile(checkpointer=checkpointer)
print("graph compiled: START -> prepare -> request_approval -> finalize -> END")


### Step 1-2: the agent process starts the purchase, and suspends


In [ ]:
thread_id = "purchase-in-progress"
cfg = {"configurable": {"thread_id": thread_id}}

result = purchase_agent.invoke(
    {"item": "20x GPU instances (1 week)", "amount": 4200.00, "request_id": "", "verdict": ""},
    cfg,
)

print(f"\ninvoke() returned: {result}")
snap = purchase_agent.get_state(cfg)
print(f"graph status: next={snap.next}  (non-empty == suspended)")
print("\nAt this point the agent process could exit entirely. Nothing is")
print("lost -- the checkpoint and the queue entry both live on disk.")


In [5]:
# ---- Proof: the pending request is genuinely durable, independent of the --
# ---- graph object still sitting in this notebook's memory ------------------

print("Reading the queue file FRESH from disk (as a separate reviewer process would):")
fresh_queue = ApprovalQueue(QUEUE_PATH)     # a brand new object, same file
for req in fresh_queue.pending():
    print(f"  [{req['request_id']}] {req['summary']}  (status={req['status']})")

Reading the queue file FRESH from disk (as a separate reviewer process would):
  [beccd3cc] Approve purchase: 20x GPU instances (1 week) for $4200.00  (status=pending)


### 5. The reviewer: a genuinely separate cell

This is the part that makes the pattern real rather than cosmetic. The next
cell does not touch `purchase_agent`, `cfg`, or any Python variable from the
cells above. It reads **only the queue file** - exactly what a reviewer's
dashboard, running in a different process on a different machine, would have
access to.

### REVIEWER SIDE -- imagine this cell runs on someone else's laptop,


In [ ]:
# ---- hours later, with no access to any variable defined above -------------

def reviewer_dashboard(queue_path: Path) -> list:
    """What a reviewer sees: a plain, human-readable list of open requests."""
    q = ApprovalQueue(queue_path)
    items = q.pending()
    print(f"{'request':<10} {'summary'}")
    print("-" * 60)
    for r in items:
        print(f"{r['request_id']:<10} {r['summary']}")
    return items


def reviewer_decide(queue_path: Path, request_id: str, approved: bool,
                    reviewer: str, reason: str = "") -> None:
    """What clicking Approve/Reject in a dashboard would call."""
    q = ApprovalQueue(queue_path)
    q.decide(request_id, approved, reviewer, reason)
    verdict = "APPROVED" if approved else "REJECTED"
    print(f"[{reviewer}] {verdict} request {request_id}: {reason or '(no reason given)'}")


print("=== Reviewer opens their dashboard ===")
open_requests = reviewer_dashboard(QUEUE_PATH)

print("\n=== Reviewer makes a decision ===")
target = open_requests[0]["request_id"]
reviewer_decide(QUEUE_PATH, target, approved=True, reviewer="finance-lead@company",
                reason="Within Q3 infra budget.")


### 6. Resuming: a separate invocation, matched by `thread_id`

The reviewer's decision is now in the queue, but the **graph** does not know
that yet - nothing has told it. Something has to notice the answer and call
`Command(resume=...)` against the right `thread_id`. In a real deployment this
is a small poller or a webhook the reviewer UI calls on submit; here it is the
next cell, deliberately written as if it had no memory of the cells above.

### The RESUMING side: a separate invocation, "later"


In [ ]:
# This function receives NOTHING from the earlier cells except what it reads
# fresh from disk (the queue) and the thread_id it is told to check -- the
# same discipline as the reviewer cell above.

def resume_if_decided(graph, queue_path: Path, thread_id: str) -> Optional[dict]:
    """Check the queue for a decision on this thread's pending request and,
    if found, resume the suspended graph with it."""
    q = ApprovalQueue(queue_path)
    snap = graph.get_state({"configurable": {"thread_id": thread_id}})
    if not snap.next:
        return None   # nothing suspended on this thread

    # Find the request this suspended thread is waiting on. A real system
    # would store request_id -> thread_id explicitly rather than scanning;
    # kept simple here since the mapping is the point, not the lookup speed.
    matching = [r for r in q._read().values()
               if r["thread_id"] == thread_id and r["status"] != "pending"]
    if not matching:
        return None   # still waiting on the human

    decision = matching[0]
    resume_value = decision["status"]      # "approved" or "rejected"
    result = graph.invoke(Command(resume=resume_value),
                          {"configurable": {"thread_id": thread_id}})
    return result


print("Checking for a decision and resuming if one exists...\n")
final = resume_if_decided(purchase_agent, QUEUE_PATH, thread_id)
print(f"\nfinal state: {final}")


### What just happened, and why it is the whole point

The cell that resumed the graph never touched `interrupt()`, never saw the
original `.invoke()` call, and does not need to run in the same process, on
the same machine, or within any particular time window of the request being
created. It found the decision by reading a file and matched it to a suspended
thread by an id. That is the entire mechanism a production system needs, in
miniature: **durable request → decoupled reviewer → resume by id.**

Try calling `resume_if_decided` again on a thread with no pending decision
yet - it correctly returns `None` rather than resuming a thread that has
nothing to resume with. That check (`if not matching: return None`) is what
lets this function be called **repeatedly, safely, from a poller** - which
is exactly the shape notebook 02's timeout logic builds on.

### Key takeaways

- "The human is watching" is the wrong default for any approval that
  represents real-world latency - the process that asks cannot stay alive
  waiting, and does not need to.
- The pattern: **enqueue the request durably, then `interrupt()`** (in that
  order - reversing it risks a suspended thread nobody can find), and let a
  **separate process**, matched only by `thread_id`, decide when and how to
  resume.
- A reviewer's decision is just data in a durable store. Nothing about acting
  on it requires touching the original graph process - proven above by
  writing the reviewer and resume logic as functions that read only from
  disk, never from earlier cells' variables.
- `Command(resume=...)` can be issued by **any** process holding the right
  `thread_id`, at any time after suspension. That is the property this whole
  module exists to demonstrate.

**Next:** `02_timeout_and_escalation.ipynb` - what happens when nobody
answers in time, and how to escalate to a different reviewer automatically.